# Tarea 2. Ingeniería de características temporales
**Entrada:** `snies_raw.csv` (Dataset 1 de la Tarea 1: 330 filas = 33 territorios × 10 años).
**Salidas:** `dataset2.csv` (tasas de crecimiento interanual por nivel de formación) y `dataset3.csv` (variación interanual del porcentaje oficial).

Fórmulas:
- Tasa de crecimiento: $g_i = \dfrac{x_{i+1}-x_i}{x_i}\times 100\ (\%)$
- Variación del porcentaje oficial: $\Delta O_i = O_{i+1}-O_i$ (puntos porcentuales)

## 1. Cargar y filtrar por departamento

In [1]:
import numpy as np
import pandas as pd

RUTA = "snies_raw.csv"          # CSV generado en la Tarea 1
df = pd.read_csv(RUTA)

NIVELES = ["tecnica", "tecnologica", "universitaria"]
COLS = [f"{a}_{a+1}" for a in range(2016, 2025)]   # 2016_2017 ... 2024_2025 (9 intervalos)

def filtrar_departamento(df, dep):
    """Devuelve las filas de un departamento ordenadas por año."""
    return df[df["departamento"] == dep].sort_values("año")

display(filtrar_departamento(df, "Bogotá, D.C."))

,departamento,año,tecnica,tecnologica,universitaria,oficial,privado
50,"Bogotá, D.C.",2016,33441,170898,507209,29.6,70.4
51,"Bogotá, D.C.",2017,31459,179321,519284,30.3,69.7
52,"Bogotá, D.C.",2018,38061,169548,525513,29.8,70.2
53,"Bogotá, D.C.",2019,33981,146778,512491,29.7,70.3
54,"Bogotá, D.C.",2020,26251,143609,472038,30.7,69.3
55,"Bogotá, D.C.",2021,29369,254058,460279,40.7,59.3
56,"Bogotá, D.C.",2022,37150,250838,457445,39.6,60.4
57,"Bogotá, D.C.",2023,40452,252203,457723,38.6,61.4
58,"Bogotá, D.C.",2024,44856,256981,463211,37.8,62.2
59,"Bogotá, D.C.",2025,45993,313531,482581,40.6,59.4


## 2. Tasas de crecimiento interanual por nivel de formación
**Tratamiento de divisiones por cero** (según las notas aclaratorias del enunciado):
1. **Nota 2 (se evalúa primero):** si en una misma fila (departamento, nivel) hay **más de dos** divisiones por cero, la fila se elimina del Dataset 2.
2. **Nota 1:** si hay una o dos, el valor faltante se estima como el **promedio de las tasas de los intervalos inmediatamente anterior y posterior**; si solo existe un intervalo vecino con tasa válida, se usa ese valor.
3. Si ninguno de los dos vecinos tiene tasa válida (casos de ceros consecutivos en el borde), no hay cómo estimar el valor y la fila también se excluye (queda registrada).

Solo se considera división por cero cuando $x_i = 0$. Si $x_{i+1}=0$ la tasa es válida (−100 %).

In [2]:
def tasas_crecimiento(x):
    """x: 10 matrículas anuales. Devuelve 9 tasas (%); NaN donde x_i == 0 (división por cero)."""
    x = np.asarray(x, dtype=float)
    with np.errstate(divide="ignore", invalid="ignore"):
        g = (x[1:] - x[:-1]) / x[:-1] * 100
    g[x[:-1] == 0] = np.nan
    return g

def estimar_faltantes(g):
    """Reemplaza cada NaN por el promedio de sus vecinos inmediatos con tasa válida (valores originales)."""
    g_est = g.copy()
    for i in np.where(np.isnan(g))[0]:
        vecinos = [g[j] for j in (i - 1, i + 1) if 0 <= j < len(g) and not np.isnan(g[j])]
        if vecinos:
            g_est[i] = np.mean(vecinos)
    return g_est

filas, imputados, excluidos = [], [], []
for dep, d in df.groupby("departamento"):
    d = d.sort_values("año")
    for nivel in NIVELES:
        g = tasas_crecimiento(d[nivel].values)
        n0 = int(np.isnan(g).sum())
        if n0 > 2:                                   # Nota 2: se evalúa antes de estimar
            excluidos.append((dep, nivel, f"{n0} divisiones por cero"))
            continue
        if n0 > 0:                                   # Nota 1: estimación
            g_est = estimar_faltantes(g)
            if np.isnan(g_est).any():
                excluidos.append((dep, nivel, "sin vecinos válidos para estimar"))
                continue
            for i in np.where(np.isnan(g))[0]:
                imputados.append((dep, nivel, COLS[i], round(g_est[i], 2)))
            g = g_est
        filas.append([dep, nivel, *g])

ds2 = pd.DataFrame(filas, columns=["departamento", "nivel"] + COLS)
ds2[COLS] = ds2[COLS].round(2)
ds2 = ds2.sort_values(["departamento", "nivel"]).reset_index(drop=True)
ds2.head(9)

,departamento,nivel,2016_2017,2017_2018,2018_2019,2019_2020,2020_2021,2021_2022,2022_2023,2023_2024,2024_2025
0,Amazonas,tecnica,-94.12,-100.00,-100.00,-100.00,6.52,113.04,-38.78,-3.33,72.41
1,Amazonas,tecnologica,-28.86,1.26,27.95,54.61,-48.35,-11.55,-12.03,10.94,2.46
2,Amazonas,universitaria,-8.43,0.95,0.31,-1.25,0.00,10.48,9.77,26.70,11.57
3,Antioquia,tecnica,-17.72,10.45,11.86,15.47,-13.00,-14.32,-5.34,-14.78,27.41
4,Antioquia,tecnologica,2.41,-2.89,-7.12,-0.51,-12.10,-8.04,-8.96,0.42,4.85
5,Antioquia,universitaria,3.31,2.91,-1.67,-4.22,7.18,-0.48,-0.38,2.24,5.80
6,Arauca,tecnologica,5.46,-26.66,-3.06,-13.32,-5.07,0.17,-6.50,7.31,37.04
7,Arauca,universitaria,-1.73,-7.05,-1.24,-10.50,-5.10,-14.63,-16.97,-12.96,-24.28
8,"Archipiélago de San Andrés, Providencia y Sant...",tecnica,195.45,15.38,45.33,82.57,361.31,-52.83,-46.88,43.04,98.48


In [3]:
print("Dataset 2:", ds2.shape, "(máximo posible: 99 filas × 11 columnas)")
print("Valores estimados por división por cero:", len(imputados))
print("Filas excluidas:", len(excluidos))
print("\nEstimaciones realizadas:")
display(pd.DataFrame(imputados, columns=["departamento", "nivel", "intervalo", "valor_estimado"]))
print("Filas excluidas del Dataset 2:")
display(pd.DataFrame(excluidos, columns=["departamento", "nivel", "motivo"]))

Dataset 2: (93, 11) (máximo posible: 99 filas × 11 columnas)
Valores estimados por división por cero: 10
Filas excluidas: 6

Estimaciones realizadas:


,departamento,nivel,intervalo,valor_estimado
0,Amazonas,tecnica,2018_2019,-100.00
1,Amazonas,tecnica,2020_2021,6.52
2,Caquetá,tecnica,2018_2019,-89.08
3,Caquetá,tecnica,2021_2022,-31.82
4,Cauca,tecnica,2021_2022,-100.00
5,Cauca,tecnica,2022_2023,5900.00
6,Cesar,tecnica,2021_2022,-43.75
7,Chocó,tecnica,2017_2018,-43.68
8,Nariño,tecnica,2021_2022,-29.02
9,Quindío,tecnica,2022_2023,-78.12


Filas excluidas del Dataset 2:


,departamento,nivel,motivo
0,Arauca,tecnica,8 divisiones por cero
1,Guainía,tecnica,5 divisiones por cero
2,Guaviare,tecnica,7 divisiones por cero
3,Putumayo,tecnica,6 divisiones por cero
4,Vaupés,tecnica,7 divisiones por cero
5,Vichada,tecnica,8 divisiones por cero


In [4]:
# Verificación manual con el ejemplo del enunciado (fórmula aplicada a mano)
ej = filtrar_departamento(df, "Bogotá, D.C.").set_index("año")["universitaria"]
print("Bogotá universitaria 2016→2017:", round((ej[2017] - ej[2016]) / ej[2016] * 100, 2), "%")
print("En Dataset 2:", ds2.query("departamento=='Bogotá, D.C.' and nivel=='universitaria'")["2016_2017"].iloc[0], "%")
assert ds2[COLS].isna().sum().sum() == 0

# Control de valores extremos (tasas > 200 %): suelen venir de bases muy pequeñas (p. ej. 1 → 60 estudiantes)
largo = ds2.melt(id_vars=["departamento", "nivel"], var_name="intervalo", value_name="tasa")
display(largo[largo["tasa"] > 200].sort_values("tasa", ascending=False).reset_index(drop=True))

Bogotá universitaria 2016→2017: 2.38 %
En Dataset 2: 2.38 %


,departamento,nivel,intervalo,tasa
0,Cauca,tecnica,2022_2023,5900.00
1,Cauca,tecnica,2023_2024,5900.00
2,Cauca,tecnica,2018_2019,815.38
3,Norte de Santander,tecnica,2024_2025,531.15
4,Vaupés,universitaria,2019_2020,486.11
5,"Archipiélago de San Andrés, Providencia y Sant...",tecnica,2020_2021,361.31
6,Casanare,tecnica,2018_2019,212.12


**Observación sobre valores extremos.** Hay tasas muy altas que no son errores de cálculo sino consecuencia de bases diminutas
(p. ej. Cauca, técnica profesional: de 1 a 60 estudiantes entre 2023 y 2024 = +5.900 %). Además, la regla de estimación del enunciado
(usar el único vecino válido) hace que ese mismo valor se repita en el intervalo anterior cuando este tenía división por cero.
Se **conservan** tal como indica el enunciado, y las Tareas 3 y 4 usan estadísticos/procedimientos robustos para que no dominen el resultado.

## 3. Variación interanual del porcentaje oficial (Dataset 3)
Una fila por departamento y nueve columnas con $\Delta O_i$ en puntos porcentuales.
Nota: `oficial` viene de la Tarea 1 redondeado a 1 decimal, por eso las variaciones tienen resolución de 0,1 pp.

In [5]:
piv = df.pivot(index="departamento", columns="año", values="oficial").sort_index(axis=1)
ds3 = piv.diff(axis=1).iloc[:, 1:]
ds3.columns = COLS
ds3 = ds3.round(2).reset_index()

print("Dataset 3:", ds3.shape, "(esperado: 33 filas × 10 columnas)")
assert ds3.shape == (33, 10) and ds3.isna().sum().sum() == 0
ds3.head()

Dataset 3: (33, 10) (esperado: 33 filas × 10 columnas)


,departamento,2016_2017,2017_2018,2018_2019,2019_2020,2020_2021,2021_2022,2022_2023,2023_2024,2024_2025
0,Amazonas,25.5,-4.6,11.8,3.6,3.8,0.2,2.6,-2.7,0.5
1,Antioquia,0.5,-0.7,0.0,-0.6,1.3,-0.3,0.6,1.8,2.6
2,Arauca,1.6,1.2,1.0,6.0,13.1,6.8,5.4,11.2,11.3
3,"Archipiélago de San Andrés, Providencia y Sant...",0.9,-1.8,0.3,-2.2,3.7,-4.3,3.4,0.0,0.9
4,Atlántico,1.6,0.9,-3.4,-0.5,-2.4,-0.5,-0.8,2.1,4.2


## 4. Exportación de los Datasets 2 y 3

In [6]:
ds2.to_csv("dataset2.csv", index=False)
print("Guardado dataset2.csv:", ds2.shape)
display(pd.read_csv("dataset2.csv").head())

ds3.to_csv("dataset3.csv", index=False)
print("Guardado dataset3.csv:", ds3.shape)
display(pd.read_csv("dataset3.csv").head())

Guardado dataset2.csv: (93, 11)


,departamento,nivel,2016_2017,2017_2018,2018_2019,2019_2020,2020_2021,2021_2022,2022_2023,2023_2024,2024_2025
0,Amazonas,tecnica,-94.12,-100.00,-100.00,-100.00,6.52,113.04,-38.78,-3.33,72.41
1,Amazonas,tecnologica,-28.86,1.26,27.95,54.61,-48.35,-11.55,-12.03,10.94,2.46
2,Amazonas,universitaria,-8.43,0.95,0.31,-1.25,0.00,10.48,9.77,26.70,11.57
3,Antioquia,tecnica,-17.72,10.45,11.86,15.47,-13.00,-14.32,-5.34,-14.78,27.41
4,Antioquia,tecnologica,2.41,-2.89,-7.12,-0.51,-12.10,-8.04,-8.96,0.42,4.85


Guardado dataset3.csv: (33, 10)


,departamento,2016_2017,2017_2018,2018_2019,2019_2020,2020_2021,2021_2022,2022_2023,2023_2024,2024_2025
0,Amazonas,25.5,-4.6,11.8,3.6,3.8,0.2,2.6,-2.7,0.5
1,Antioquia,0.5,-0.7,0.0,-0.6,1.3,-0.3,0.6,1.8,2.6
2,Arauca,1.6,1.2,1.0,6.0,13.1,6.8,5.4,11.2,11.3
3,"Archipiélago de San Andrés, Providencia y Sant...",0.9,-1.8,0.3,-2.2,3.7,-4.3,3.4,0.0,0.9
4,Atlántico,1.6,0.9,-3.4,-0.5,-2.4,-0.5,-0.8,2.1,4.2
